In [ ]:
import numpy as np
import pandas as pd
from typing import Callable, Dict, List
from collections import defaultdict
from einops import einsum

np.random.seed(3)

class ProbTable:
    def __init__(self, description: str, data, shape: tuple = None):
        if isinstance(data, Callable):
            self.probs = np.empty(shape)
            def recurse(assignment: list):
                if len(assignment) == len(shape):
                    self.probs[tuple(assignment)] = data(*assignment)
                else:
                    for i in range(shape[len(assignment)]):
                        recurse(assignment + [i])
            recurse([])
        else:
            self.probs = np.array(data)

    @property
    def p(self) -> np.ndarray:
        return self.probs

    

In [4]:
# Telephone - Rejection Sampling
# A -> B -> C

def Bernoulli(prob: float) -> int:
    return np.random.choice([0, 1], p=[1 - prob, prob])

def normalize_dict(counts: Dict) -> Dict:
    total_counts = sum(counts.values())
    if total_counts == 0: return {}
    return {k: v / total_counts for k, v in counts.items()}

def telephone_program():
    A = Bernoulli(0.5)
    B = Bernoulli(0.8 if A == 1 else 0.2)
    C = Bernoulli(0.8 if B == 1 else 0.2)
    return {"A": A, "B": B, "C": C}

def rejection_sampling(program: Callable, evidence: Callable, query: Callable, num_samples: int = 1000):
    counts = defaultdict(int)
    for _ in range(num_samples):
        sample = program()
        if evidence(sample):
            counts[query(sample)] += 1
    return normalize_dict(counts)

# P(A = 1 | C = 1)
evidence = lambda s: s["C"] == 1
query = lambda s: s["A"]
results = rejection_sampling(telephone_program, evidence, query, num_samples=10000)
print(f"Rejection sampling result P(A=1 | C=1):{results.get(1, 0):.4f}")

Rejection sampling result P(A=1 | C=1):0.6780


In [7]:
# Telephone - Gibbs Sampling
# A -> B -> C

def sample_dict(probs: Dict) -> int:
    return np.random.choice(list(probs.keys()), p=list(probs.values()))

def gibbs_sampling(x: Dict, vars: list, query: Callable, joint_probs: Callable, num_iters: int = 100):
    counts = defaultdict(int)
    for _ in range(num_iters):
        for var in vars:
            # 1. compute joint distribution when value of var is 0 or 1
            probs = {}
            for value in [0, 1]:
                probs[value] = joint_probs(x, var, value)
            # 2. Normalize and sample var from the props
            probs = normalize_dict(probs)
            x[var] = sample_dict(probs)
            # 3. Count value count of the query
            counts[query(x)] += 1
    return normalize_dict(counts)

# local conditional variables
p_a = ProbTable("A", [0.5, 0.5]) # P(A)
p_b_given_a = ProbTable("B | A", lambda a, b: 0.8 if a == b else 0.2, shape=(2, 2)) # P(B | A)
p_c_given_b = ProbTable("C | B", lambda b, c: 0.8 if b == c else 0.2, shape=(2, 2)) # P(C | B)

def full_joint_probs(x: Dict, var: str, value: int) -> float:
    y = x | {var: value} # overwrite the value of var in x
    a, b, c = y["A"], y["B"], y["C"]
    # compute joint distribution P(A=a, B=b, C=c)
    return p_a.p[a] * p_b_given_a.p[a, b] * p_c_given_b.p[b, c]

# P(A = 1 | C = 1)
x = {"A": 1, "B": 0, "C": 1}
query = lambda s: s["A"]
gibbs_result = gibbs_sampling(x, ["A", "B"], query, full_joint_probs, num_iters=10000)
print(f"Gibbs sampling result P(A=1 | C=1):{gibbs_result.get(1, 0):.4f}")

Gibbs sampling result P(A=1 | C=1):0.6767


In [9]:
# Telephone - Gibbs Sampling with Markov Blanket
# A -> B -> C

# compute markov blanket joint probs for this case: # P(A = 1 | C = 1)
def markov_blanket_joint_probs(x: Dict, var: str, value: int) -> float:
    y = x | {var: value}
    a, b, c = y["A"], y["B"], y["C"]
    if var == "A":
        # P(A) * P(B | A), ignore P(C | B)
        return p_a.p[a] * p_b_given_a.p[a, b]
    elif var == "B":
        # P(B | A) * P(C | B), ignore P(A)
        return p_b_given_a.p[a, b] * p_c_given_b.p[b, c]
    else:
        raise ValueError(f"unknown variable: {var}")

# P(A = 1 | C = 1)
mb_results = gibbs_sampling(x, ["A", "B"], query, markov_blanket_joint_probs, num_iters=2000)
print(f"Gibbs sampling with Markov Blanket result P(A=1 | C=1):{mb_results.get(1, 0):.4f}")

Gibbs sampling with Markov Blanket result P(A=1 | C=1):0.6810


In [13]:
# Rejection Sampling is hard when evidence is low probability
# e.g. P(A | B = 1)，but P(B=1) is rare
def rare_event_program():
    A = Bernoulli(0.5)
    B = Bernoulli(0.0001 if A == 0 else 0.0002) # P(B=1) is rare
    return {"A": A, "B": B}

res_rej_rare = rejection_sampling(rare_event_program, lambda s: s["A"], lambda s: s["B"]==1, num_samples=2000)
print(f"[Rare evidence] Rejection sampling results:{res_rej_rare}")

# Gibbs Sampling is hard when variables are highly related
# e.g. A -> B, and A=B
p_a_corr = ProbTable("A", [0.5, 0.5])
p_b_corr = ProbTable("B | A", lambda a, b: float(a == b), shape=(2, 2))

def corr_joint_prob(x, var, val):
    y = x | {var: val}
    return p_a_corr.p[y["A"]] * p_b_corr.p[y["A"], y["B"]]

# P (A=1)
# If init x is {A:0, B:0}, Gibbs sampling can never sample {A:1, B:1}
x = {"A": 0, "B": 0}
query = lambda s: s["A"]
res_gibbs_corr = gibbs_sampling(x, ["A", "B"], query, corr_joint_prob, num_iters=100)
print(f"[High correlation] Gibbs sample results for P(A=1):{res_gibbs_corr.get(1, 0)}")


[Rare evidence] Rejection sampling results:{np.False_: 1.0}
[High correlation] Gibbs sample results for P(A=1):0
